In [1]:
import sys

# Branch, tag or commit to benchmark. The commit that actually got installed is
# read back in the next cell and recorded, so a branch name is fine here.
# main can carry the same version number as the last release, so pip would
# think it is already installed and skip it. Hence the force-reinstall.
LFG_REF = "main"

!{sys.executable} -m pip install -q --force-reinstall --no-cache-dir \
  "git+https://github.com/JuliaYershova/llm-feature-gen.git@{LFG_REF}"
!{sys.executable} -m pip install -q scikit-learn catboost transformers torch pandas opencv-python scipy tabulate

# Restart the kernel before running the next cell.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import csv
import json
import os
import random
import shutil
import ssl
import subprocess
import sys
import tarfile
import threading
import time
import urllib.request
from importlib.metadata import distribution
from pathlib import Path

import numpy as np
import pandas as pd

# --- model -------------------------------------------------------------------
# One open-weights VLM for all four modalities, served by Ollama on this machine.
# Must be the -instruct tag: the "thinking" variants spend their whole token
# budget reasoning and return empty content, which the library reports as
# invalid JSON. Talking to Ollama directly, not through a proxy, so the GPU we
# measure below is the GPU that does the work.
#MODEL = "qwen3-vl:32b-instruct"
MODEL = "qwen3.8:27b"
OLLAMA_URL = "http://localhost:11434"

os.environ["LOCAL_OPENAI_BASE_URL"] = f"{OLLAMA_URL}/v1"
os.environ["LOCAL_OPENAI_API_KEY"] = "ollama"   # ignored by Ollama, required by the SDK
os.environ["LOCAL_MODEL_VISION"] = MODEL
os.environ["LOCAL_MODEL_TEXT"] = MODEL

# --- experiment --------------------------------------------------------------
SEED = 46
RUNS = 3                     # repeats per modality, each with its own discovery sample
DISCOVERY_SAMPLES = 3        # items per class shown to the discovery step
EVAL_SAMPLES = 60            # items per class used for evaluation
TEMPERATURE = 0.0            # the library default; greedy, so runs can be repeated
DISCOVERY_FRAMES = 3         # frames per clip shown to the discovery step
NUM_FRAMES = 8               # frames sampled per video during generation
USE_AUDIO = False            # most UCF-101 clips have no usable audio track
MIN_FEATURES = 12
FOLDS, REPEATS = 5, 10       # repeated stratified CV used for all scoring

# Four deliberately confusable classes per modality, except Adult, which is
# binary by construction.
# Birds: four near-identical black species (picked via CUB's own human-annotated
# attributes, so the choice is independent of any model).
IMAGE_CLASSES = ["029.American_Crow", "030.Fish_Crow",
                 "107.Common_Raven", "027.Shiny_Cowbird"]
VIDEO_CLASSES = ["BreastStroke", "FrontCrawl", "Diving", "CliffDiving"]
TEXT_CLASSES = ["World", "Sports", "Business", "SciTech"]
TABULAR_CLASSES = ["at_most_50k", "above_50k"]

# Adult rows are serialised into one sentence per row; this is the column the
# library reads for tabular generation.
TABULAR_TEXT_COLUMN = "record"

# --- electricity -------------------------------------------------------------
# Local inference has no API fee, so "cost" means electricity.
ELECTRICITY_PRICE_CZK = 6.50
CZK_TO_USD = 23.5

GPU_INDICES = [0, 1, 2]

# --- library -----------------------------------------------------------------
import llm_feature_gen

# pip records the commit it installed from git. That commit, not "main", is
# what the paper cites, and it names the output folder, so a run on a newer
# version never overwrites an older one.
installed = json.loads(distribution("llm-feature-gen").read_text("direct_url.json") or "{}")
LFG_COMMIT = installed.get("vcs_info", {}).get("commit_id", llm_feature_gen.__version__)

# --- paths -------------------------------------------------------------------
RAW_DIR = Path("raw_data")
RAW_CUB = RAW_DIR / "CUB_200_2011"
RAW_UCF = RAW_DIR / "UCF-101"
RAW_AGNEWS = RAW_DIR / "ag_news"
RAW_ADULT = RAW_DIR / "adult"

DATA_DIR = Path("data/benchmark")
#OUT_DIR = Path(f"outputs/benchmark_seed{SEED}_{LFG_COMMIT[:7]}")
OUT_DIR = Path(f"outputs/benchmark_seed{SEED}_{LFG_COMMIT[:7]}_{MODEL.replace(':', '-')}")

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}
TEXT_EXTS = {".txt"}
TABULAR_EXTS = {".csv"}

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("config ok —", MODEL, "| llm-feature-gen", llm_feature_gen.__version__,
      "@", LFG_COMMIT[:7], "->", OUT_DIR)

config ok — qwen3.8:27b | llm-feature-gen 0.1.14 @ 21b6408 -> outputs/benchmark_seed46_21b6408_qwen3.8-27b


In [3]:
def ollama(path):
    """Ollama's native API, which says more about the model than /v1 does."""
    with urllib.request.urlopen(f"{OLLAMA_URL}{path}", timeout=10) as response:
        return json.load(response)


# Confirms Ollama is up on this machine and that MODEL is spelled the way it is
# tagged locally, before anything long starts. A tag can be re-pulled with
# different weights, so the digest and quantisation go into the report too.
served = {m["name"]: m for m in ollama("/api/tags")["models"]}
assert MODEL in served, f"{MODEL} is not served here. Pull it with: ollama pull {MODEL}"

ENVIRONMENT = {
    "ollama": ollama("/api/version")["version"],
    "model_digest": served[MODEL]["digest"],
    "quantization": served[MODEL]["details"]["quantization_level"],
    "gpu": subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader", "-i", str(GPU_INDICES[0])],
        capture_output=True, text=True).stdout.strip(),
    "python": sys.version.split()[0],
}
print(ENVIRONMENT)

{'ollama': '0.34.1', 'model_digest': 'aaee06c39dcf2437cde036998d960e1fc1494b8191be7cc9657d01e509097813', 'quantization': 'Q4_K_M', 'gpu': 'NVIDIA L40S', 'python': '3.12.6'}


In [4]:
import platform
import tempfile


def download(url, dest, insecure=False, attempts=100):
    """Download a file, resuming after a dropped connection.

    Three workarounds live here: some hosts reject urllib's default User-Agent
    with a 403, crcv.ucf.edu serves a certificate chain that doesn't validate,
    and it also cuts long transfers partway through, so we re-request with a
    Range header until the file is complete.

    Writes to a .part file first, so an interrupted download never leaves
    behind a truncated archive that looks finished.
    """
    dest = Path(dest)
    part = dest.with_suffix(dest.suffix + ".part")
    context = ssl._create_unverified_context() if insecure else None
    total = None

    for attempt in range(1, attempts + 1):
        have = part.stat().st_size if part.exists() else 0
        if total is not None and have >= total:
            break

        headers = {"User-Agent": "Mozilla/5.0"}
        if have:
            headers["Range"] = f"bytes={have}-"

        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, context=context, timeout=60) as response:
                if have and response.status != 206:   # server ignored Range: start over
                    part.unlink()
                    have, total = 0, None
                if total is None:
                    total = have + int(response.headers.get("Content-Length", 0))
                with open(part, "ab") as f:
                    shutil.copyfileobj(response, f)
        except Exception as e:
            have = part.stat().st_size if part.exists() else 0
            print(f"  attempt {attempt}: dropped at {have / 1e9:.2f} GB ({type(e).__name__}), resuming")
            continue

        if part.stat().st_size >= total:
            break
        print(f"  attempt {attempt}: {part.stat().st_size / 1e9:.2f} / {total / 1e9:.2f} GB, resuming")

    have = part.stat().st_size
    if total is not None and have < total:
        raise RuntimeError(
            f"{dest.name} is still incomplete: {have / 1e9:.2f} of {total / 1e9:.2f} GB. "
            "Re-run the cell to carry on from here."
        )
    part.replace(dest)
    return dest


def ensure_7zz():
    """Return a path to a 7-Zip binary that can extract .rar."""
    if platform.system() == "Windows":
        for candidate in (
            Path(r"C:\Program Files\7-Zip\7z.exe"),
            Path(r"C:\Program Files (x86)\7-Zip\7z.exe"),
        ):
            if candidate.exists():
                return candidate
        raise RuntimeError("7-Zip not found. Install it: winget install 7zip.7zip")

    # Linux/macOS: grab a static build, no system package needed.
    exe = Path(tempfile.gettempdir()) / "7zz"
    if not exe.exists():
        url = "https://github.com/ip7z/7zip/releases/download/26.02/7z2602-linux-x64.tar.xz"
        tarball = download(url, Path(tempfile.gettempdir()) / "7zz.tar.xz")
        with tarfile.open(tarball) as tar, open(exe, "wb") as dst:
            shutil.copyfileobj(tar.extractfile("7zz"), dst)
        exe.chmod(0o755)
    return exe


def extract_rar(archive, dest_dir):
    """Extract a .rar with 7-Zip, surfacing its output when it fails."""
    result = subprocess.run(
        [str(ensure_7zz()), "x", str(archive), f"-o{dest_dir}", "-y"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"7-Zip failed on {archive.name} (exit {result.returncode}).\n"
            f"{result.stdout}\n{result.stderr}\n"
            "If the archive is truncated, delete it and re-run to download again."
        )


def ensure_cub():
    if (RAW_CUB / "images").exists():
        print("CUB already present")
        return
    RAW_CUB.parent.mkdir(parents=True, exist_ok=True)
    archive = RAW_CUB.parent / "CUB_200_2011.tgz"
    if not archive.exists():
        print("downloading CUB-200-2011 (~1.1 GB)...")
        download("https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz",
                 archive)
    print("extracting CUB...")
    with tarfile.open(archive) as tar:
        tar.extractall(RAW_CUB.parent)


def ensure_ucf():
    if RAW_UCF.exists():
        print("UCF already present")
        return
    RAW_UCF.parent.mkdir(parents=True, exist_ok=True)
    archive = RAW_UCF.parent / "UCF101.rar"
    if not archive.exists():
        print("downloading UCF-101 (~6.9 GB, takes a while)...")
        download("https://www.crcv.ucf.edu/data/UCF101/UCF101.rar", archive, insecure=True)
    print("extracting UCF...")
    extract_rar(archive, RAW_UCF.parent)


def ensure_agnews():
    """AG News training split: label (1-4), title, description, no header."""
    csv_path = RAW_AGNEWS / "train.csv"
    if csv_path.exists():
        print("AG News already present")
        return
    RAW_AGNEWS.mkdir(parents=True, exist_ok=True)
    print("downloading AG News (~29 MB)...")
    download("https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/"
             "master/data/ag_news_csv/train.csv", csv_path)


def ensure_adult():
    """Adult census income, via OpenML so we get column names and no manual parsing."""
    csv_path = RAW_ADULT / "adult.csv"
    if csv_path.exists():
        print("Adult already present")
        return
    from sklearn.datasets import fetch_openml

    RAW_ADULT.mkdir(parents=True, exist_ok=True)
    print("downloading Adult...")
    data = fetch_openml("adult", version=2, as_frame=True)
    df = data.frame.dropna(subset=["class"])
    df.to_csv(csv_path, index=False)


ensure_cub()
ensure_ucf()
ensure_agnews()
ensure_adult()

print("CUB classes:", len(list((RAW_CUB / "images").iterdir())))
print("UCF classes:", len([p for p in RAW_UCF.iterdir() if p.is_dir()]))
print("AG News rows:", sum(1 for _ in open(RAW_AGNEWS / "train.csv", encoding="utf-8")))
print("Adult rows:", len(pd.read_csv(RAW_ADULT / "adult.csv")))

CUB already present
UCF already present
AG News already present
Adult already present
CUB classes: 200
UCF classes: 101
AG News rows: 120000
Adult rows: 48842


In [5]:
def split_for_runs(items, rng=None):
    """One discovery sample per run, then a single evaluation set.

    The discovery samples are disjoint and the evaluation set is shared, so
    repeated runs vary the sample the schema is induced from while every run is
    scored on exactly the same items.
    """
    if rng is not None:
        rng.shuffle(items)
    reserved = DISCOVERY_SAMPLES * RUNS
    samples = [items[i * DISCOVERY_SAMPLES:(i + 1) * DISCOVERY_SAMPLES] for i in range(RUNS)]
    return samples, items[reserved:reserved + EVAL_SAMPLES]


def build_modality(src_root, out_base, classes, exts, seed=SEED):
    """Lay the data out the way the library expects.

    discovery/run<n>/  a flat, mixed sample the model sees once to propose a schema
    generation/<class> the items we actually evaluate on

    Discovery and evaluation never overlap, so a schema is not proposed from
    the items it is later scored on.
    """
    shutil.rmtree(out_base, ignore_errors=True)
    rng = random.Random(seed)

    for class_name in classes:
        files = sorted(f for f in (src_root / class_name).iterdir()
                       if f.suffix.lower() in exts)
        samples, evaluation = split_for_runs(files, rng)

        for run, sample in enumerate(samples):
            discovery_dir = out_base / "discovery" / f"run{run}"
            discovery_dir.mkdir(parents=True, exist_ok=True)
            for f in sample:
                shutil.copy2(f, discovery_dir / f"{class_name}_{f.name}")

        class_dir = out_base / "generation" / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        for f in evaluation:
            shutil.copy2(f, class_dir / f.name)

        print(f"  {class_name}: {DISCOVERY_SAMPLES} discovery x {RUNS} runs "
              f"/ {len(evaluation)} eval")


def build_text(csv_path, out_base, classes, seed=SEED):
    """AG News rows become one .txt per document, so text uses the same layout.

    Labels in the file are 1-4 and follow the order of `classes`; the fourth
    class is Sci/Tech, renamed here only because it becomes a folder name.
    """
    shutil.rmtree(out_base, ignore_errors=True)

    news = pd.read_csv(csv_path, header=None, names=["label", "title", "description"])
    # The published CSV uses a literal backslash-n as an in-field line break.
    news["text"] = (news["title"].str.strip() + ". "
                    + news["description"].str.strip()).str.replace("\\", " ", regex=False)

    for label, class_name in enumerate(classes, start=1):
        subset = news[news["label"] == label].sample(
            DISCOVERY_SAMPLES * RUNS + EVAL_SAMPLES, random_state=seed)
        samples, evaluation = split_for_runs(list(subset.itertuples()))

        for run, sample in enumerate(samples):
            discovery_dir = out_base / "discovery" / f"run{run}"
            discovery_dir.mkdir(parents=True, exist_ok=True)
            for row in sample:
                (discovery_dir / f"{class_name}_{row.Index}.txt").write_text(
                    row.text, encoding="utf-8")

        class_dir = out_base / "generation" / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        # Row index in the filename keeps every item traceable to its source row.
        for row in evaluation:
            (class_dir / f"agnews_{row.Index}.txt").write_text(row.text, encoding="utf-8")

        print(f"  {class_name}: {DISCOVERY_SAMPLES} discovery x {RUNS} runs "
              f"/ {len(evaluation)} eval")


def build_tabular(csv_path, out_base, classes, seed=SEED):
    """Adult rows, split the same way, but kept as CSVs.

    The library reads tabular data row by row from one file per class, so the
    layout is discovery/run<n>/discovery.csv plus generation/<class>/<class>.csv.
    Each row is also serialised into one sentence in TABULAR_TEXT_COLUMN, which
    is what the LLM reads; the original columns stay for the baseline.
    """
    shutil.rmtree(out_base, ignore_errors=True)

    adult = pd.read_csv(csv_path).fillna("missing")
    # OpenML labels: "<=50K" and ">50K", in the order of `classes`.
    label_map = dict(zip(sorted(adult["class"].unique()), classes))
    adult["class_name"] = adult["class"].map(label_map)

    fields = [c for c in adult.columns if c not in ("class", "class_name")]
    adult[TABULAR_TEXT_COLUMN] = adult[fields].apply(
        lambda row: ", ".join(f"{name} {value}" for name, value in row.items()), axis=1)

    discovery_rows = [[] for _ in range(RUNS)]
    for class_name in classes:
        subset = adult[adult["class_name"] == class_name].sample(
            DISCOVERY_SAMPLES * RUNS + EVAL_SAMPLES, random_state=seed)
        samples, evaluation = split_for_runs(list(range(len(subset))))

        for run, sample in enumerate(samples):
            discovery_rows[run].append(subset.iloc[sample])

        class_dir = out_base / "generation" / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        subset.iloc[evaluation].to_csv(class_dir / f"{class_name}.csv", index=False)

        print(f"  {class_name}: {DISCOVERY_SAMPLES} discovery x {RUNS} runs "
              f"/ {len(evaluation)} eval")

    for run, rows in enumerate(discovery_rows):
        discovery_dir = out_base / "discovery" / f"run{run}"
        discovery_dir.mkdir(parents=True, exist_ok=True)
        pd.concat(rows).to_csv(discovery_dir / "discovery.csv", index=False)


print("CUB:")
build_modality(RAW_CUB / "images", DATA_DIR / "cub", IMAGE_CLASSES, IMAGE_EXTS)
print("UCF:")
build_modality(RAW_UCF, DATA_DIR / "ucf", VIDEO_CLASSES, VIDEO_EXTS)
print("AG News:")
build_text(RAW_AGNEWS / "train.csv", DATA_DIR / "agnews", TEXT_CLASSES)
print("Adult:")
build_tabular(RAW_ADULT / "adult.csv", DATA_DIR / "adult", TABULAR_CLASSES)

CUB:
  029.American_Crow: 3 discovery x 3 runs / 51 eval
  030.Fish_Crow: 3 discovery x 3 runs / 51 eval
  107.Common_Raven: 3 discovery x 3 runs / 50 eval
  027.Shiny_Cowbird: 3 discovery x 3 runs / 51 eval
UCF:
  BreastStroke: 3 discovery x 3 runs / 60 eval
  FrontCrawl: 3 discovery x 3 runs / 60 eval
  Diving: 3 discovery x 3 runs / 60 eval
  CliffDiving: 3 discovery x 3 runs / 60 eval
AG News:
  World: 3 discovery x 3 runs / 60 eval
  Sports: 3 discovery x 3 runs / 60 eval
  Business: 3 discovery x 3 runs / 60 eval
  SciTech: 3 discovery x 3 runs / 60 eval
Adult:
  at_most_50k: 3 discovery x 3 runs / 60 eval
  above_50k: 3 discovery x 3 runs / 60 eval


In [6]:
from llm_feature_gen.providers import LocalProvider

# The provider counts tokens itself: usage_summary() reads the totals,
# reset_usage() clears them between phases.
provider = LocalProvider(
    default_vision_model=MODEL,
    default_text_model=MODEL,
    temperature=TEMPERATURE,
)

create = provider.client.chat.completions.create
provider.client.chat.completions.create = (
    lambda **kwargs: create(**kwargs, extra_body={"reasoning_effort": "none"}))

# Sanity check before committing to a long run: if the model is misconfigured
# we want to know in two seconds, not twenty minutes. It also pulls the model
# into VRAM, which the GPU detection in the next cells relies on.
print(provider.text_features(["Say hi in JSON."], feature_gen=True))
# What will actually be sent. A typo in the env vars silently falls back to llama3.
print(provider.text_model, provider.vision_model, provider.temperature, provider.max_tokens)

# How much the loaded model can read at once (None on older Ollama versions).
# Video discovery sends 36 frames, so this has to be well above the prompt
# tokens printed for each discovery.
ENVIRONMENT["context_length"] = next(
    (m.get("context_length") for m in ollama("/api/ps")["models"] if m["name"] == MODEL), None)
print("context length:", ENVIRONMENT["context_length"])

print(provider.usage_summary())

[{'greeting': 'hi'}]
qwen3.8:27b qwen3.8:27b 0.0 2048
context length: 262144
{'calls': 1, 'prompt_tokens': 132, 'completion_tokens': 13, 'total_tokens': 145}


In [7]:
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold

from llm_feature_gen import load_discovered_features

# One fold split, reused by every model we score, so the scores are paired.
# Repeated, because a single 5-fold split of ~240 items gives a noisy mean, and
# refitting a classifier costs nothing next to generating the features.
CV = RepeatedStratifiedKFold(n_splits=FOLDS, n_repeats=REPEATS, random_state=SEED)


def schema_structure_validity(schema_path):
    """Is the proposed schema itself well-formed?

    Checks that every discovered feature has a name, a description and a
    non-empty list of possible values. This is about the discovery output,
    not about what the model later fills in.
    """
    features = load_discovered_features(schema_path).get("proposed_features", [])
    if not features:
        return 0, 0.0
    well_formed = sum(
        1 for f in features
        if isinstance(f, dict)
        and "feature" in f
        and "description" in f
        and isinstance(f.get("possible_values"), list)
        and f["possible_values"]
    )
    return len(features), well_formed / len(features) * 100


def take(X, rows):
    """Subset whatever a baseline uses: an array, a list of texts, a frame."""
    if isinstance(X, pd.DataFrame):
        return X.iloc[rows]
    if isinstance(X, list):
        return [X[i] for i in rows]
    return X[rows]


def classify_cv(X, y, make_model=None):
    """Per-fold macro-F1 on the shared CV split.

    Takes a factory rather than an estimator and fits the folds by hand:
    cross_val_score clones the estimator, and CatBoost cannot be cloned.
    Returns every fold, not just the mean, so we can report a spread and run a
    paired test on the same folds. Generated features always go through logistic
    regression: we're measuring how much signal they carry, not how strong a
    model we can fit on top of them.
    """
    make_model = make_model or (lambda: LogisticRegression(max_iter=2000, random_state=SEED))
    scores = []
    for train, test in CV.split(np.zeros(len(y)), y):
        model = make_model()
        model.fit(take(X, train), y[train])
        predicted = np.asarray(model.predict(take(X, test))).ravel()
        scores.append(f1_score(y[test], predicted, average="macro"))
    return np.array(scores)


def corrected_ttest(ours_folds, base_folds):
    """Paired t-test over CV folds, corrected for overlapping training sets.

    Any two training sets in k-fold CV share most of their items, so the fold
    scores are not independent and a plain paired t-test is overconfident.
    Nadeau & Bengio (2003) inflate the variance by the test/train size ratio;
    Bouckaert & Frank (2004) recommend this for repeated k-fold CV.
    """
    d = np.asarray(ours_folds) - np.asarray(base_folds)
    k = len(d)
    t = d.mean() / np.sqrt((1 / k + 1 / (FOLDS - 1)) * d.var(ddof=1))
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return {"t": round(float(t), 3), "p": round(float(p), 4)}

In [8]:
def gpu_uuid_to_index():
    """UUID to index, read once: the mapping doesn't change while we run."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=uuid,index", "--format=csv,noheader"],
        capture_output=True, text=True)
    return {u.strip(): int(i) for u, i in
            (line.split(",") for line in result.stdout.strip().splitlines())}


GPU_UUIDS = gpu_uuid_to_index()


def compute_apps():
    """GPUs currently running a compute process, as {gpu_index: [pids]}."""
    result = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=gpu_uuid,pid", "--format=csv,noheader"],
        capture_output=True, text=True)
    apps = {}
    for line in result.stdout.strip().splitlines():
        uuid, pid = (p.strip() for p in line.split(","))
        index = GPU_UUIDS.get(uuid)
        if index is not None:
            apps.setdefault(index, []).append(int(pid))
    return apps


def resolve_gpus():
    """Bill only the cards the model actually sits on.

    Summing every GPU in the box would add someone else's job, or just noise,
    to our electricity figure. Run this after the provider sanity check, so the
    model is loaded and visible.
    """
    if GPU_INDICES:
        return list(GPU_INDICES)
    busy = sorted(compute_apps())
    if not busy:
        raise RuntimeError(
            "No compute process visible on any GPU. Either the model was "
            "unloaded — re-run the provider check — or it is not served from "
            "this machine, in which case the energy numbers here mean nothing. "
            "Set GPU_INDICES by hand to override."
        )
    return busy


def read_power(gpus):
    """Current power draw, in watts, for the GPUs we're billing."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,power.draw", "--format=csv,noheader,nounits"],
        capture_output=True, text=True)
    readings = {int(parts[0]): float(parts[1])
                for parts in (line.split(",") for line in result.stdout.strip().splitlines())}
    return {g: w for g, w in readings.items() if g in gpus}


def measure_idle(gpus, samples=8, interval=2):
    """Baseline draw with nothing of ours running, so we can subtract it later.

    Call this immediately before a run, not once at the top of the notebook:
    on a shared card the baseline moves.
    """
    readings = {}
    for _ in range(samples):
        for gpu, watts in read_power(gpus).items():
            readings.setdefault(gpu, []).append(watts)
        time.sleep(interval)
    idle = {gpu: round(float(np.mean(v)), 2) for gpu, v in readings.items()}
    print("idle power (W):", idle)
    return idle


class GPULogger:
    """Sample GPU power in the background and turn it into kWh and money.

    Also records how many compute processes were on the card at each sample.
    Power is per-GPU, not per-process, so a run that shared the card with
    someone else can't be corrected afterwards — only spotted and thrown away.
    """

    def __init__(self, path, gpus, interval=2):
        self.path = Path(path)
        self.gpus = list(gpus)
        self.interval = interval
        self._stop = threading.Event()

    def _loop(self):
        with open(self.path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "gpu_id", "power_W", "n_procs"])
            while not self._stop.is_set():
                stamp = round(time.time(), 2)
                apps = compute_apps()
                for gpu, watts in read_power(self.gpus).items():
                    writer.writerow([stamp, gpu, watts, len(apps.get(gpu, []))])
                f.flush()
                self._stop.wait(self.interval)   # wakes up at once when stopped

    def __enter__(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self.stopped = time.time()
        self._thread.join()

    def energy(self, idle_power):
        """Energy above idle, converted to CZK and USD.

        Each sample counts until the next one, by the clock: the two nvidia-smi
        calls per loop make the real spacing longer than `interval`, so
        multiplying by `interval` would undercount.
        """
        log = pd.read_csv(self.path)
        watt_hours = 0.0
        for gpu, rows in log.groupby("gpu_id"):
            above_idle = (rows["power_W"] - idle_power[gpu]).clip(lower=0).to_numpy()
            seconds = np.diff(rows["timestamp"].to_numpy(), append=self.stopped)
            watt_hours += (above_idle * seconds).sum() / 3600
        kwh = watt_hours / 1000
        czk = kwh * ELECTRICITY_PRICE_CZK
        return {"kWh": round(kwh, 5),
                "cost_CZK": round(czk, 3),
                "cost_USD": round(czk / CZK_TO_USD, 5),
                # a process that wasn't on the card when we started isn't ours
                "new_procs": int(log["n_procs"].max() - log["n_procs"].iloc[0]),
                "gpus": sorted(log["gpu_id"].unique().tolist())}


GPUS = resolve_gpus()
print("billing GPUs:", GPUS)

# Worth one look: anything on the card besides Ollama is someone else's job,
# and its draw ends up in our idle baseline.
!nvidia-smi

billing GPUs: [0, 1, 2]
Sat Oct 24 17:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:03:00.0 Off |                    0 |
| N/A   38C    P0             95W /  350W |   15389MiB /  46068MiB |     22%      Default |
|                                         |                        |                  N/A |
+-----------------------

In [9]:
import cv2
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def embed_images(images, batch_size=32):
    """L2-normalised CLIP embeddings for a list of PIL images."""
    batches = []
    for i in range(0, len(images), batch_size):
        inputs = processor(images=images[i:i + batch_size], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            features = clip.get_image_features(**inputs)
        # newer transformers return an output object instead of a plain tensor
        if not torch.is_tensor(features):
            features = features.pooler_output
        batches.append((features / features.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.concatenate(batches)


def read_frames(path, num_frames=NUM_FRAMES):
    """Uniformly sample frames from a clip."""
    capture = cv2.VideoCapture(str(path))
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, max(total - 1, 0),
                          num=min(num_frames, max(total, 1)), dtype=int)
    frames = []
    for index in indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = capture.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    capture.release()
    return frames


def embed_video(path):
    """Avg-CLIP: mean of the frame embeddings, renormalised."""
    frames = embed_images(read_frames(path))
    mean = frames.mean(axis=0)
    return mean / np.linalg.norm(mean)


def collect(generation_dir, exts):
    """Every evaluation item with its ground-truth class.

    Returns the identifiers the library will write into the File column, so a
    generated table can be lined up with the baseline item by item.
    """
    paths, labels = [], []
    for class_dir in sorted(p for p in generation_dir.iterdir() if p.is_dir()):
        for f in sorted(class_dir.iterdir()):
            if f.suffix.lower() in exts:
                paths.append(f)
                labels.append(class_dir.name)
    return paths, [p.name for p in paths], np.array(labels)


def collect_tabular(generation_dir):
    """Adult rows, with the same three return values as collect().

    The library identifies a generated row as "<file name>__row_<index>", so we
    build the same identifier here from the row's position in its class file.
    """
    frames, ids, labels = [], [], []
    for class_dir in sorted(p for p in generation_dir.iterdir() if p.is_dir()):
        csv_path = next(class_dir.glob("*.csv"))
        rows = pd.read_csv(csv_path)
        frames.append(rows)
        ids += [f"{csv_path.name}__row_{i}" for i in range(len(rows))]
        labels += [class_dir.name] * len(rows)
    return pd.concat(frames, ignore_index=True), ids, np.array(labels)


print("CLIP ready on", DEVICE)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP ready on cuda


In [10]:
from catboost import CatBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

# Baselines run first, on purpose: CLIP would otherwise be drawing power during
# the energy measurement and inflate the cost of the LLM pipeline.
# Each baseline is a factory, not a fitted model: classify_cv builds a fresh one
# per fold, because CatBoost cannot be cloned the way sklearn expects.

image_paths, image_ids, image_labels = collect(DATA_DIR / "cub" / "generation", IMAGE_EXTS)
image_X = embed_images([Image.open(p).convert("RGB") for p in image_paths])
image_baseline = ("CLIP", None)

video_paths, video_ids, video_labels = collect(DATA_DIR / "ucf" / "generation", VIDEO_EXTS)
video_X = np.stack([embed_video(p) for p in video_paths])
video_baseline = ("Avg-CLIP", None)

# Text: raw documents, vectorised inside the pipeline so a fold never sees
# vocabulary from its own test half.
text_paths, text_ids, text_labels = collect(DATA_DIR / "agnews" / "generation", TEXT_EXTS)
text_X = [p.read_text(encoding="utf-8") for p in text_paths]
text_baseline = ("TF-IDF", lambda: make_pipeline(
    TfidfVectorizer(), LogisticRegression(max_iter=2000, random_state=SEED)))

# Tabular: the original columns. Numerics stay numeric, the rest go to CatBoost
# as categories with missing values named explicitly.
tabular_rows, tabular_ids, tabular_labels = collect_tabular(DATA_DIR / "adult" / "generation")
tabular_fields = [c for c in tabular_rows.columns
                  if c not in ("class", "class_name", TABULAR_TEXT_COLUMN)]

tabular_X = tabular_rows[tabular_fields].copy()
numeric_fields = list(tabular_X.select_dtypes(include="number").columns)
categorical_fields = [c for c in tabular_fields if c not in numeric_fields]
tabular_X[categorical_fields] = (tabular_X[categorical_fields]
                                 .astype("object").fillna("missing").astype(str))

tabular_baseline = ("CatBoost", lambda: CatBoostClassifier(
    cat_features=categorical_fields, verbose=0, random_seed=SEED))

for name, X, y, (baseline_name, factory) in [
    ("Images ", image_X, image_labels, image_baseline),
    ("Videos ", video_X, video_labels, video_baseline),
    ("Text   ", text_X, text_labels, text_baseline),
    ("Tabular", tabular_X, tabular_labels, tabular_baseline),
]:
    folds = classify_cv(X, y, factory)
    print(f"{name} — {baseline_name:9s} macro-F1 = {folds.mean():.3f} ± {folds.std():.3f}"
          f"  ({len(y)} items)")

Images  — CLIP      macro-F1 = 0.545 ± 0.065  (203 items)
Videos  — Avg-CLIP  macro-F1 = 0.963 ± 0.026  (240 items)
Text    — TF-IDF    macro-F1 = 0.685 ± 0.057  (240 items)
Tabular — CatBoost  macro-F1 = 0.659 ± 0.087  (120 items)


In [11]:
# Free the card before anything is measured: an idle CLIP still occupies VRAM
# and shows up as a second compute process on the GPU we are about to bill.
del clip, processor
torch.cuda.empty_cache()
print("CLIP released; compute processes now:", compute_apps())

CLIP released; compute processes now: {0: [2572886, 2572411], 1: [1770856, 2572886], 2: [2572886]}


In [12]:
from llm_feature_gen import (
    discover_features_multiclass,
    discover_features_multiclass_images,
    discover_features_multiclass_tabular,
    discover_features_multiclass_videos,
    generate_features,
)

# Everything a modality needs to be discovered, generated and scored. Built
# after the baselines, because that is where the item order and the reference
# scores come from.
MODALITIES = {
    "image": dict(data=DATA_DIR / "cub", classes=IMAGE_CLASSES,
                  ids=image_ids, labels=image_labels,
                  baseline_name="CLIP", baseline_X=image_X, baseline_model=None),
    "video": dict(data=DATA_DIR / "ucf", classes=VIDEO_CLASSES,
                  ids=video_ids, labels=video_labels,
                  baseline_name="Avg-CLIP", baseline_X=video_X, baseline_model=None),
    "text": dict(data=DATA_DIR / "agnews", classes=TEXT_CLASSES,
                 ids=text_ids, labels=text_labels,
                 baseline_name="TF-IDF", baseline_X=text_X,
                 baseline_model=text_baseline[1]),
    "tabular": dict(data=DATA_DIR / "adult", classes=TABULAR_CLASSES,
                    ids=tabular_ids, labels=tabular_labels,
                    baseline_name="CatBoost", baseline_X=tabular_X,
                    baseline_model=tabular_baseline[1]),
}

# What the library writes when none of a feature's listed values applies. It
# also writes it in place of any answer that is not on the list.
NOT_GIVEN = "not given by LLM"


def discover(modality, run, schema_path):
    """Propose a schema from this run's discovery sample."""
    spec = MODALITIES[modality]
    sample = spec["data"] / "discovery" / f"run{run}"
    common = dict(classes=spec["classes"], provider=provider, min_features=MIN_FEATURES,
                  output_dir=schema_path.parent, output_filename=schema_path.name)

    if modality == "image":
        discover_features_multiclass_images(image_paths_or_folder=sample, **common)
    elif modality == "video":
        clips = len(spec["classes"]) * DISCOVERY_SAMPLES
        discover_features_multiclass_videos(
            videos_or_folder=sample,
            num_frames=DISCOVERY_FRAMES, use_audio=USE_AUDIO,
            max_videos_to_sample=clips,
            # By default the library keeps 15 frames and picks them by
            # clustering, which can leave whole clips out. Send all of them.
            max_total_frames_payload=clips * DISCOVERY_FRAMES,
            random_seed=SEED, **common)
    elif modality == "text":
        discover_features_multiclass(texts_or_file=sample, **common)
    else:
        discover_features_multiclass_tabular(
            file_or_folder=sample / "discovery.csv",
            text_column=TABULAR_TEXT_COLUMN, **common)


def generate(modality, schema_path):
    """Fill the schema in for every evaluation item."""
    spec = MODALITIES[modality]
    options = {}
    if modality == "video":
        options = {"use_audio": USE_AUDIO, "num_frames": NUM_FRAMES}
    elif modality == "tabular":
        options = {"text_column": TABULAR_TEXT_COLUMN}

    generate_features(
        root_folder=spec["data"] / "generation",
        discovered_features_path=schema_path,
        classes=spec["classes"],
        provider=provider,
        output_dir=schema_path.parent,
        merge_to_single_csv=True,
        merged_csv_name="all_features.csv",
        failure_threshold=None,      # don't abort a long run on a few failures
        **options,
    )


def measured(phase, out, idle_power, work):
    """Run one phase under the power logger: its time, tokens and energy."""
    provider.reset_usage()
    started = time.time()
    with GPULogger(out / f"gpu_{phase}.csv", GPUS) as log:
        work()
    return {"time_s": round(time.time() - started, 1),
            "tokens": provider.usage_summary(), **log.energy(idle_power)}


def load_run(modality, run):
    """The generated table of one run, indexed by item, and its schema."""
    out = OUT_DIR / f"{modality}_run{run}"
    table = pd.read_csv(out / "all_features.csv").drop_duplicates("File").set_index("File")
    return table, load_discovered_features(out / "discovered_features.json")


def run_pipeline(modality, run=0):
    """Discover a feature schema and fill it in for every evaluation item.

    Scoring is a separate step, so the analysis can be redone from the saved
    tables without touching the GPU. Power and tokens are logged per phase:
    discovery is a one-off cost, generation is what grows with the data.
    Idle is measured immediately before the run, because the card is shared
    and the baseline moves.
    """
    out = OUT_DIR / f"{modality}_run{run}"
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    schema_path = out / "discovered_features.json"

    # Ollama drops an idle model out of VRAM. Wake it before measuring, or the
    # idle baseline is taken without the model and the reload lands in our bill.
    provider.text_features(["ping"], feature_gen=True)
    time.sleep(10)   # let the card drop out of its busy power state first
    idle_power = measure_idle(GPUS)

    discovery = measured("discovery", out, idle_power,
                         lambda: discover(modality, run, schema_path))
    n_features, structure_validity = schema_structure_validity(schema_path)
    tokens = discovery["tokens"]["prompt_tokens"] + discovery["tokens"]["completion_tokens"]
    print(f"discovered {n_features} features (structure validity {structure_validity:.0f}%), "
          f"{tokens} tokens of context {ENVIRONMENT['context_length']}")
    if ENVIRONMENT["context_length"] and tokens > 0.9 * ENVIRONMENT["context_length"]:
        print("WARNING: discovery nearly filled the context, Ollama may have cut the prompt")

    generation = measured("generation", out, idle_power,
                          lambda: generate(modality, schema_path))

    ids = MODALITIES[modality]["ids"]
    table, _ = load_run(modality, run)
    n_generated = len(set(table.index) & set(ids))
    assert n_generated, f"no {modality} item made it through generation, see the errors above"

    record = {
        "modality": modality,
        "run": run,
        "n_features": n_features,
        "schema_structure_validity_%": round(structure_validity, 1),
        "n_generated": n_generated,
        "n_failed": len(ids) - n_generated,
        "idle_power_W": idle_power,
        "discovery": discovery,
        "generation": generation,
        # per 100 items, generation only: discovery is a one-off, reported above
        "time_per_100_s": round(generation["time_s"] / n_generated * 100, 1),
        "cost_per_100_USD": round(generation["cost_USD"] / n_generated * 100, 5),
    }
    # On disk as well, so a dead kernel doesn't take hours of runs with it.
    (out / "run.json").write_text(json.dumps(record, indent=2))
    return record


def answer_breakdown(table, schema):
    """What the model answered, before the library tidied it up.

    The library swaps any answer that is not on a feature's list for
    NOT_GIVEN before saving the table, so the table itself always looks
    conformant. The model's own answers are kept in raw_llm_output. Each cell
    is a listed value, NOT_GIVEN chosen by the model itself (abstained), or
    something else that the library then replaced.
    """
    listed = {}
    for f in schema["proposed_features"]:
        if isinstance(f, dict) and f.get("feature"):
            listed.setdefault(f["feature"], f.get("allowed_values") or f.get("possible_values") or [])

    counts = {"listed": 0, "abstained": 0, "replaced": 0}
    for raw in table["raw_llm_output"]:
        answers = json.loads(raw)["features"]
        for feature, values in listed.items():
            answer = answers.get(feature)
            if not values or answer in values:
                counts["listed"] += 1
            elif answer == NOT_GIVEN:
                counts["abstained"] += 1
            else:
                counts["replaced"] += 1
    total = sum(counts.values())
    return {f"{key}_%": round(n / total * 100, 2) for key, n in counts.items()}


def score_modality(modality, records):
    """Score the given runs of one modality against its baseline.

    Every run is scored on the items that came through generation in all of
    them, on the same folds. So the baseline is one set of numbers, and the
    runs can be averaged fold by fold for a single test per modality.
    """
    spec = MODALITIES[modality]
    tables = [load_run(modality, r["run"]) for r in records]

    survived = set.intersection(*(set(table.index) for table, _ in tables))
    keep = [i for i, item in enumerate(spec["ids"]) if item in survived]
    items, y = [spec["ids"][i] for i in keep], spec["labels"][keep]

    base_folds = classify_cv(take(spec["baseline_X"], keep), y, spec["baseline_model"])

    runs, ours_folds = [], []
    for record, (table, schema) in zip(records, tables):
        features = [c for c in table.columns if c not in ("Class", "raw_llm_output")]
        X = pd.get_dummies(table.loc[items, features].astype(str), dtype=float).values
        folds = classify_cv(X, y)
        ours_folds.append(folds)
        runs.append({**record, **answer_breakdown(table, schema),
                     "test": corrected_ttest(folds, base_folds),
                     "macro_f1_ours": round(float(folds.mean()), 4),
                     "folds_ours": folds.round(4).tolist()})

    return {
        "baseline": spec["baseline_name"],
        "n_items": len(items),
        "macro_f1_baseline": round(float(base_folds.mean()), 4),
        "folds_baseline": base_folds.round(4).tolist(),
        # our side of the test: each fold's score averaged over the runs
        "test": corrected_ttest(np.mean(ours_folds, axis=0), base_folds),
        "runs": runs,
    }

In [13]:
# One run of each first, cheapest modality first, so a configuration mistake
# shows up after minutes instead of after the video run.
results = {}
for modality in ("text", "tabular", "image", "video"):
    results[modality] = [run_pipeline(modality, run=0)]

for modality, records in results.items():
    s = score_modality(modality, records)
    r = s["runs"][0]
    print(f'{modality:8s} {r["macro_f1_ours"]:.3f} vs {s["baseline"]} {s["macro_f1_baseline"]:.3f}'
          f'  | replaced {r["replaced_%"]:.1f}%, abstained {r["abstained_%"]:.1f}%'
          f'  | failed {r["n_failed"]}')

idle power (W): {0: 78.22, 1: 85.51, 2: 84.75}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/text_run0/discovered_features.json
discovered 14 features (structure validity 100%), 2749 tokens of context 262144


SciTech: 100%|██████████| 60/60 [03:05<00:00,  3.10s/file]


idle power (W): {0: 80.53, 1: 88.43, 2: 91.21}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/tabular_run0/discovered_features.json
discovered 13 features (structure validity 100%), 2212 tokens of context 262144


above_50k: 100%|██████████| 1/1 [02:26<00:00, 146.94s/file]


idle power (W): {0: 80.4, 1: 87.56, 2: 91.32}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/image_run0/discovered_features.json
discovered 14 features (structure validity 100%), 3030 tokens of context 262144


027.Shiny_Cowbird: 100%|██████████| 51/51 [02:09<00:00,  2.54s/file]


idle power (W): {0: 80.33, 1: 87.58, 2: 90.81}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/video_run0/discovered_features.json
discovered 12 features (structure validity 100%), 4766 tokens of context 262144


CliffDiving: 100%|██████████| 60/60 [02:52<00:00,  2.87s/file]


text     0.765 vs TF-IDF 0.685  | replaced 0.5%, abstained 0.0%  | failed 0
tabular  0.647 vs CatBoost 0.659  | replaced 0.5%, abstained 0.0%  | failed 0
image    0.461 vs CLIP 0.545  | replaced 0.4%, abstained 0.0%  | failed 0
video    0.772 vs Avg-CLIP 0.963  | replaced 0.0%, abstained 0.0%  | failed 0


In [14]:
# Repeats. Each run discovers its schema from its own disjoint sample, while
# the evaluation items and folds stay fixed and decoding is greedy. So the
# spread across runs shows how much the result depends on the sample.
for run in range(1, RUNS):
    for modality in results:
        results[modality].append(run_pipeline(modality, run=run))
    print(f"run {run} done")

idle power (W): {0: 78.22, 1: 86.27, 2: 85.0}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/text_run1/discovered_features.json
discovered 14 features (structure validity 100%), 2450 tokens of context 262144


SciTech: 100%|██████████| 60/60 [02:37<00:00,  2.63s/file]


idle power (W): {0: 80.55, 1: 88.47, 2: 91.4}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/tabular_run1/discovered_features.json
discovered 14 features (structure validity 100%), 2352 tokens of context 262144


above_50k: 100%|██████████| 1/1 [02:39<00:00, 159.54s/file]


idle power (W): {0: 80.48, 1: 87.7, 2: 91.15}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/image_run1/discovered_features.json
discovered 14 features (structure validity 100%), 3069 tokens of context 262144


027.Shiny_Cowbird: 100%|██████████| 51/51 [02:16<00:00,  2.68s/file]


... (7 outputs left out)

027.Shiny_Cowbird: 100%|██████████| 51/51 [01:52<00:00,  2.21s/file]


idle power (W): {0: 80.66, 1: 88.56, 2: 90.92}
Features saved to outputs/benchmark_seed46_21b6408_qwen3.8-27b/video_run2/discovered_features.json
discovered 12 features (structure validity 100%), 4790 tokens of context 262144


CliffDiving: 100%|██████████| 60/60 [02:57<00:00,  2.96s/file]

run 2 done


In [15]:
DATASETS = {
    "text": "AG News (4)",
    "tabular": "Adult (2)",
    "image": "CUB-200 (4)",
    "video": "UCF-101 (4)",
}

# Read back from the run folders rather than taken from memory, so this cell
# also works in a fresh kernel once the runs are done.
results = {m: [json.loads((OUT_DIR / f"{m}_run{r}" / "run.json").read_text()) for r in range(RUNS)]
           for m in MODALITIES}
scores = {m: score_modality(m, results[m]) for m in MODALITIES}


def summarize(modality):
    """One table row per modality: mean across runs, with the spread that
    justifies running them."""
    s = scores[modality]
    runs = s["runs"]

    def mean(key):
        return float(np.mean([r[key] for r in runs]))

    def sd(key):
        return float(np.std([r[key] for r in runs], ddof=1)) if len(runs) > 1 else 0.0

    for r in runs:
        for phase in ("discovery", "generation"):
            if r[phase]["new_procs"]:
                print(f'WARNING: another process joined the GPU during {modality} '
                      f'run {r["run"]} {phase}; its cost is inflated')

    return {
        "Modality": modality.title(),
        "Dataset": DATASETS[modality],
        "Runs": len(runs),
        "#F": f'{mean("n_features"):.1f}',
        "Schema validity": f'{mean("schema_structure_validity_%"):.0f}%',
        "Value conformity": f'{100 - mean("replaced_%"):.1f}%',
        "Not given": f'{mean("abstained_%") + mean("replaced_%"):.1f}%',
        "Baseline": s["baseline"],
        "Base": f'{s["macro_f1_baseline"]:.3f}',
        "Ours": f'{mean("macro_f1_ours"):.3f} ± {sd("macro_f1_ours"):.3f}',
        "p": f'{s["test"]["p"]:.3f}',
        "p per run": " / ".join(f'{r["test"]["p"]:.3f}' for r in runs),
        "Items": s["n_items"],
        "Failed": sum(r["n_failed"] for r in runs),
        "Discovery s": f'{np.mean([r["discovery"]["time_s"] for r in runs]):.0f}',
        "s/100": f'{mean("time_per_100_s"):.0f}',
        "USD/100": f'{mean("cost_per_100_USD"):.4f}',
    }


report = {
    "model": MODEL,
    "library_version": llm_feature_gen.__version__,
    "library_commit": LFG_COMMIT,
    "environment": ENVIRONMENT,
    "gpus_billed": GPUS,
    "config": {
        "seed": SEED, "runs": RUNS, "temperature": TEMPERATURE,
        "max_tokens": provider.max_tokens, "min_features": MIN_FEATURES,
        "discovery_samples_per_class": DISCOVERY_SAMPLES,
        "eval_samples_per_class": EVAL_SAMPLES,
        "discovery_frames": DISCOVERY_FRAMES, "generation_frames": NUM_FRAMES,
        "use_audio": USE_AUDIO, "cv": f"{REPEATS}x{FOLDS}-fold stratified",
    },
    "modalities": scores,
}
(OUT_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
(OUT_DIR / "pip_freeze.txt").write_text(subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)

table = pd.DataFrame([summarize(m) for m in ("text", "tabular", "image", "video")])
table.to_csv(OUT_DIR / "table.csv", index=False)

print(table.to_string(index=False))
print()
print(table.to_markdown(index=False))

Modality     Dataset  Runs   #F Schema validity Value conformity Not given Baseline  Base          Ours     p             p per run  Items  Failed Discovery s s/100 USD/100
    Text AG News (4)     3 14.0            100%            99.7%      0.3%   TF-IDF 0.685 0.789 ± 0.022 0.001 0.021 / 0.003 / 0.000    240       0          20   285  0.0053
 Tabular   Adult (2)     3 13.7            100%            99.8%      0.6% CatBoost 0.659 0.654 ± 0.011 0.903 0.810 / 0.875 / 0.811    120       0          16   251  0.0047
   Image CUB-200 (4)     3 13.3            100%            99.5%      0.5%     CLIP 0.545 0.516 ± 0.051 0.417 0.074 / 0.703 / 0.614    203       0          15   249  0.0041
   Video UCF-101 (4)     3 12.0            100%           100.0%      0.0% Avg-CLIP 0.963 0.767 ± 0.025 0.000 0.000 / 0.000 / 0.000    240       0          17   305  0.0052

| Modality   | Dataset     |   Runs |   #F | Schema validity   | Value conformity   | Not given   | Baseline   |   Base | Ours        

In [16]:
def replaced_answers(modality, run=0):
    """Per feature: what the model answered outside its own list.

    These are the answers the library swapped for NOT_GIVEN. Near-misses in
    spelling or case point at the strict matching, new labels at a schema that
    doesn't fit some class.
    """
    table, schema = load_run(modality, run)
    answers = pd.DataFrame([json.loads(raw)["features"] for raw in table["raw_llm_output"]])
    for f in schema["proposed_features"]:
        values = f.get("allowed_values") or f.get("possible_values") or []
        if not values:
            continue
        column = answers[f["feature"]]
        off = column[~column.isin(values + [NOT_GIVEN])]
        print(f'\n{f["feature"]}')
        print("  listed:   ", values)
        print("  not given:", int((column == NOT_GIVEN).sum()))
        print("  replaced: ", off.value_counts().head(5).to_dict() or "none")


def class_name_values(modality, run=0):
    """Listed values that just restate a class, like front_crawl for FrontCrawl."""
    def squash(s):
        return "".join(ch for ch in str(s).lower() if ch.isalnum())

    classes = {squash(c.split(".")[-1]) for c in MODALITIES[modality]["classes"]}
    _, schema = load_run(modality, run)
    return [(f["feature"], v) for f in schema["proposed_features"]
            for v in f.get("possible_values") or [] if squash(v) in classes]


for modality in MODALITIES:
    print(modality, [class_name_values(modality, run) for run in range(RUNS)])

replaced_answers("video")

image [[], [], []]
video [[], [], []]
text [[], [], []]
tabular [[], [], []]

body_orientation
  listed:    ['horizontal', 'vertical', 'not_applicable']
  not given: 0
  replaced:  none

limb_movement_pattern
  listed:    ['alternating_arms', 'simultaneous_arms', 'no_movement', 'not_applicable']
  not given: 0
  replaced:  none

head_position
  listed:    ['submerged', 'above_water', 'not_applicable']
  not given: 0
  replaced:  none

trajectory_direction
  listed:    ['horizontal', 'vertical_down', 'not_applicable']
  not given: 0
  replaced:  none

starting_environment
  listed:    ['pool_surface', 'high_platform', 'rock_formation', 'not_applicable']
  not given: 0
  replaced:  none

water_interaction_type
  listed:    ['continuous_propulsion', 'single_impact', 'not_applicable']
  not given: 0
  replaced:  none

action_duration
  listed:    ['sustained', 'brief', 'not_applicable']
  not given: 0
  replaced:  none

presence_of_lane_markers
  listed:    ['visible', 'not_visible', 'not_

In [17]:
from sklearn.metrics import mutual_info_score

def feature_report(modality, run=0):
    """Per feature: values used, share of the commonest one, and what it says about the class."""
    table, schema = load_run(modality, run)
    labels = dict(zip(MODALITIES[modality]["ids"], MODALITIES[modality]["labels"]))
    table = table[table.index.isin(labels)]
    y = table.index.map(labels)
    rows = []
    for f in schema["proposed_features"]:
        x = table[f["feature"]].astype(str)
        rows.append({"feature": f["feature"],
                     "values used": x.nunique(),
                     "top value %": round(x.value_counts(normalize=True).iloc[0] * 100),
                     "MI (bits)": round(mutual_info_score(y, x) / np.log(2), 3)})
    return pd.DataFrame(rows).sort_values("MI (bits)", ascending=False)

for modality in MODALITIES:
    print(f"\n=== {modality}")
    print(feature_report(modality).to_string(index=False))


=== image
                 feature  values used  top value %  MI (bits)
              beak_shape            3           67      0.730
      body_size_relative            3           67      0.653
              tail_shape            5           65      0.645
          beak_thickness            3           67      0.629
  throat_feather_texture            3           68      0.613
    wing_length_relative            4           67      0.589
         body_proportion            3           70      0.586
              head_shape            2           71      0.584
               leg_color            3           92      0.166
          plumage_luster            3           51      0.118
              body_color            3           97      0.061
              head_color            3           96      0.061
               eye_color            2           99      0.016
plumage_color_uniformity            2           99      0.006

=== video
                   feature  values used  top val

In [18]:
table, _ = load_run("text", 0)
print(pd.crosstab(table["primary_subject_domain"], table["Class"]))

table, _ = load_run("video", 0)
print(pd.crosstab(table["arm_motion_pattern"], table["Class"]))

KeyError: 'primary_subject_domain'

In [19]:
import base64, io
from PIL import Image

for size in (256, 512, 1024):
    buffer = io.BytesIO()
    Image.new("RGB", (size, size), "gray").save(buffer, "JPEG")
    provider.reset_usage()
    provider.image_features([base64.b64encode(buffer.getvalue()).decode()],
                            prompt="Describe the image in JSON.", feature_gen=True)
    print(size, provider.usage_summary()["prompt_tokens"])

256 118
512 310
1024 1078
